<a href="https://colab.research.google.com/github/SashaNasonova/burnSeverity/blob/main/BARC_SBS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Burn Severity Mapping Notebook
This notebook is intended to be used for small scale, interactive burn severity mapping of individual fires. For large scale semi-automated mapping please refer to the main python scripts (https://github.com/SashaNasonova/burnSeverity).

The methodology is based on the Burned Area Reflectance Classification (BARC) product developed by the USGS that aims to estimate burn severity through a spectral comparison of pre- and post-fire medium resolution (20 - 30m) satellite imagery.

Healthy vegetation reflects strongly in the near-infrared (NIR) portion of the electromagnetic spectrum whereas rock and bare soil reflects strongly in the mid to shortwave infrared (SWIR) portion. In other words, healthy vegetation reflects strongly in NIR and reflects weakly in SWIR **(↑NIR,↓SWIR)**, whereas soil, bare rock and burned woody vegetation reflect strongly in SWIR and weakly in NIR **(↑SWIR,↓NIR)**. This inverse relationship can be leveraged to provide an estimate of burn severity where both pre- and post-fire imagery is available.

The Normalized Burn Ratio (NBR) is a spectral index that captures the relationship between NIR and SWIR bands. The difference between pre- and post-fire NBR (dNBR) can then be used to quantify wildfire burn severity (**↑dNBR ∝ ↑Severity**) using the following equations.

(1) NBR = (NIR - SWIR) / (NIR + SWIR) \\
(2) dNBR = NBRpre - NBRpost

Once dNBR has been calculated, it can be transformed into a burn severity classification product using a variety of methods ranging from simple thresholding to more complex supervised classifications informed by ground observations. This process is based on the USGS BARC256 methodology which scales the data to an 8-bit representation and utilizes static thresholds (76,110,187) to create a burn severity classification from the dNBR raster.

This notebook is divided into the following sections:

1. Set up (data import, package installation, library loading)
2. Google Earth Engine authentication and initialization
3. Fire perimeter import and visualization
4. Individual fire perimeter selection (1 fire perimeter)
5. Sensor selection
6. Pre-fire image search, visualization and selection
7. Post-fire image search, visualization and selection
8. BARC mapping
9. Final visualization for quality control
10. Image download
11. Quicklooks and area burned by severity class
12. Final export

Before beginning, please ensure that you are registered to use Google Earth Engine and have the Google Earth Engine API enabled as part of a Google Cloud project. If that is not the case, please follow the instructions on getting started (https://github.com/SashaNasonova/burnSeverity/blob/main/Getting_Started_with_GEE.md). You will need to take note of the project id and enter it later on.



---



### **Part 1: Set-up**

In [1]:
# Clone github repository to be able to access the test data and provincial extent vector data
!git clone https://github.com/SashaNasonova/burnSeverity.git

Cloning into 'burnSeverity'...
remote: Enumerating objects: 608, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 608 (delta 27), reused 9 (delta 2), pack-reused 556 (from 2)
Receiving objects: 100% (608/608), 270.26 MiB | 18.85 MiB/s, done.
Resolving deltas: 100% (297/297), done.
Updating files: 100% (242/242), done.


In [2]:
# Install the libraries
%pip install geemap
%pip install pycrs rasterio python-pptx cartopy requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 45.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.5 MB/s eta 0:00:00
  Created wheel for pycrs: filename=PyCRS-1.0.2-py3-none-any.whl size=32686 sha256=ce822b845e3828cf30df778ca7abd0f8dcefb560fd44140a35e1c9e420f58a4e
  Stored in directory: /root/.cache/pip/wheels/b5/4a/72/1ba05f57ddf2cc80ad21a26512097762561d646ff3ff85f729
Successfully built pycrs


In [3]:
# Import the libraries
import ee
import geemap
import os, json, shutil
import geopandas
from osgeo import gdal
from google.colab import files
import requests, zipfile

import rasterio
from rasterio.plot import show
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from pptx import Presentation
from pptx.util import Cm, Inches
from pptx.util import Pt

### Define functions
Please inspect the hidden cells by clicking on the sideways arrow by the Define Functions heading and run. The functions were hidden to improve the readibility of the notebook. The hidden cells contain the processing functions for mosaicking, calculating NBR, tiling for export as well as visualization and quality control.

In [4]:
## Define functions
# Helper function to get files, non-recursive
def getfiles(d,ext):
  paths = []
  for file in os.listdir(d):
      if file.endswith(ext):
          paths.append(os.path.join(d, file))
  return(paths)

# Helper function to get image acquisition date and format into ("yyyy-mm-dd")
def getDate(im):
  return(ee.Image(im).date().format("YYYY-MM-dd"))

# Helper function to get scene ids
def getSceneIds(im):
  return(ee.Image(im).get('PRODUCT_ID'))

# Functions to mosaic by image date
def mosaicByDate(indate):
  d = ee.Date(indate)
  #print(d)
  im = col.filterBounds(poly).filterDate(d, d.advance(1, "day")).mosaic()
  #print(im)
  return(im.set("system:time_start", d.millis(), "system:index", d.format("YYYY-MM-dd")))

def runDateMosaic(col_list):
  #get a list of unique dates within the list
  date_list = col_list.map(getDate).getInfo()
  udates = list(set(date_list))
  udates.sort()
  udates_ee = ee.List(udates)

  #mosaic images by unique date
  mosaic_imlist = udates_ee.map(mosaicByDate)
  return(ee.ImageCollection(mosaic_imlist))

# Calculate NBR using Sentinel-2 imagery
def NBR_S2(image):
  nbr = image.expression(
      '(NIR - SWIR) / (NIR + SWIR)', {
          'NIR': image.select('B8'),
          'SWIR': image.select('B12')}).rename('nbr')
  return(nbr)

# Calculate NBR using Landsat imagery, all bands 30m
# Handles Landsat-5, 7, 8 and 9
def NBR_Landsat(image,dattype):
  if (dattype == 'L5')|(dattype == 'L7'):
      nbr = image.expression(
          '(NIR - SWIR) / (NIR + SWIR)', {
              'NIR': image.select('SR_B4'),
              'SWIR': image.select('SR_B7')}).rename('nbr')
  elif (dattype == 'L8')|(dattype == 'L9'):
      nbr = image.expression(
          '(NIR - SWIR) / (NIR + SWIR)', {
              'NIR': image.select('SR_B5'),
              'SWIR': image.select('SR_B7')}).rename('nbr')
  else:
      print('Incorrect Landsat sensor specified')
  return(nbr)

# NBR calculation for Landsat TOA, all bands 30m
def NBR_Landsat_TOA(image,dattype):
  if (dattype == 'L5_TOA')|(dattype == 'L7_TOA'):
      nbr = image.expression(
          '(NIR - SWIR) / (NIR + SWIR)', {
              'NIR': image.select('B4'),
              'SWIR': image.select('B7')}).rename('nbr')
  elif (dattype == 'L8_TOA')|(dattype == 'L9_TOA'):
      nbr = image.expression(
          '(NIR - SWIR) / (NIR + SWIR)', {
              'NIR': image.select('B5'),
              'SWIR': image.select('B7')}).rename('nbr')
  else:
      print('Incorrect Landsat sensor specified')
  return(nbr)

# Tiling function, uses a geometry (footprint) to split into a defined
# number or rows and columns (nx,ny)
def grid_footprint(footprint,nx,ny):
  from shapely.geometry import Polygon, LineString, MultiPolygon
  from shapely.ops import split

  #polygon = footprint
  polygon = Polygon(footprint['coordinates'][0])
  #polygon = Polygon(footprint)

  minx, miny, maxx, maxy = polygon.bounds
  dx = (maxx - minx) / nx  # width of a small part
  dy = (maxy - miny) / ny  # height of a small part

  horizontal_splitters = [LineString([(minx, miny + i*dy), (maxx, miny + i*dy)]) for i in range(ny)]
  vertical_splitters = [LineString([(minx + i*dx, miny), (minx + i*dx, maxy)]) for i in range(nx)]
  splitters = horizontal_splitters + vertical_splitters

  result = polygon
  for splitter in splitters:
      result = MultiPolygon(split(result, splitter))

  coord_list = [list(part.exterior.coords) for part in result.geoms]

  poly_list = []
  for cc in coord_list:
      p = ee.Geometry.Polygon(cc)
      poly_list.append(p)
  return(poly_list)

# Applies scaling functors for surface reflectance Landsat imagery
def apply_scale_factors_ls(image):
  opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(opticalBands, None, True).addBands(thermalBands, None, True)

# Multiplies Sentinel-2 imagery by 0.0001
def apply_scale_factors_s2(image):
  opticalBands = image.select('B.*').multiply(0.0001)
  return image.addBands(opticalBands, None, True)

### QA Functions ###
def ql_3band(outshp,imgpath,outpath):
  # Get fire vector
  dfs_sub = geopandas.read_file(outshp)

  # Read 8-bit image
  src = rasterio.open(imgpath)

  # Create pre-fire quicklook image with burned area boundary overlay
  fig, ax = plt.subplots(figsize=(8, 8)) #10,13
  base = show(src,ax=ax)
  dfs_sub.plot(ax=base, edgecolor='purple', facecolor='none',linewidth=2)
  ax.axis('off')
  plt.savefig(outpath, bbox_inches='tight', dpi=300)

  plt.close()
  src = None

def generate_legend_labels(arr):
  #sort array
  arr.sort()

  default_labels = {
      0: {"black":"Unknown"},
      1: {"gray":"Unburned"},
      2: {"yellow":"Low"},
      3: {"orange":"Medium"},
      4: {"red":"High"},
      5: {"blue":"Water"}
  }

  legend_labels = {}

  for value in arr:
      #print(default_labels[value]) #for debug
      legend_labels.update(default_labels[value])

  return(legend_labels)


def ql_barc(outshp,barcpath,imgpath,outpath):
  dfs_sub = geopandas.read_file(outshp)

  # Read in BARC classification
  src = rasterio.open(barcpath)
  unique_values = np.unique(src.read(1))

  # Read in post-fire image
  src1 = rasterio.open(imgpath)

  #drop no data 9
  arr = unique_values[unique_values!=9]

  # Create cmap dictionary
  colormap_dict = {0: 'black',
                    1: 'gray',
                    2: 'yellow',
                    3: 'orange',
                    4: 'red',
                    5: 'blue'}

  colormap_unique = [colormap_dict[value] for value in arr]
  colormap = matplotlib.colors.ListedColormap(colormap_unique)

  # Create pre-fire quicklook image with burned area boundary overlay
  fig, ax = plt.subplots(figsize=(8, 8))
  background = show((src1,1),ax=ax,cmap='gray')
  base = show(src,cmap=colormap,ax=ax)
  dfs_sub.plot(ax=base, edgecolor='purple', facecolor='none',linewidth=2)
  ax.axis('off')

  legend_labels = generate_legend_labels(arr)

  patches = [Patch(color=color, label=label)
              for color, label in legend_labels.items()]

  ax.legend(handles=patches,
            bbox_to_anchor=(1.05, 1),loc='upper left',
            borderaxespad=0., facecolor="white")


  plt.savefig(outpath, bbox_inches='tight', dpi=300)

  plt.close()
  src = None
  src1 = None

# Generate quicklooks
def generate_ql(outshp,tif_file,qcdir):
  name = Path(tif_file).stem + '.png'
  ql = os.path.join(qcdir,name)
  ql_3band(outshp,tif_file,ql)
  return(ql)

def create_ppt(pptpath):
  prs = Presentation()
  prs.slide_width = Inches(16)
  prs.slide_height = Inches(9)

  prs.save(pptpath)

def add_slide(pptpath,i,j,k,l,df):
  prs = Presentation(pptpath)
  title_only_slide_layout = prs.slide_layouts[5]
  slide = prs.slides.add_slide(title_only_slide_layout)
  title = slide.shapes.title
  title.width = Cm(35)
  title.height = Cm(3.3)
  title.top = Cm(0)
  title.left = Cm(0)

  #Add title
  name = Path(k).stem
  title.text = name

  #First image (pre-fire)
  slide.shapes.add_picture(
      i, left=Cm(0.44), top=Cm(8.2), width=Cm(12.24), height=None
  )
  #Second image (post_fire)
  slide.shapes.add_picture(
      j, left=Cm(12.78), top=Cm(8.2), width=Cm(12.24), height=None
  )

  #get image height
  picture = slide.shapes[1] #second shape pre-img, first is the title
  height_cm = picture.height.cm
  #print(height_cm)

  #Third image (barc)
  slide.shapes.add_picture(
      k, left=Cm(25.02), top=Cm(8.2), width=None, height=Cm(height_cm)
  )

  #Fourth image (location map)
  slide.shapes.add_picture(
      l, left=Cm(32.5), top=Cm(0), width=None, height=Cm(7)
  )

  #add table
  rows, cols = df.shape
  left = Cm(0.9)
  top = Cm(3.2)
  width = Cm(12.0)
  height = Cm(3.75)

  #add table
  shape = slide.shapes.add_table(rows + 1, cols, left, top, width, height)
  table = shape.table

  #assign table style
  tbl =  shape._element.graphic.graphicData.tbl
  style_id = '{C083E6E3-FA7D-4D7B-A595-EF9225AFEA82}'
  tbl[0][-1].text = style_id


  # Set column names
  for col, column_name in enumerate(df.columns):
      cell = table.cell(0, col)
      cell.text = column_name
      cell.text_frame.paragraphs[0].runs[0].font.size = Pt(11)

  # Populate data
  for row in range(rows):
      for col in range(cols):
          cell = table.cell(row + 1, col)
          cell.text = str(df.iloc[row, col])

  # Change font size
  for row in table.rows:
      for cell in row.cells:
          for paragraph in cell.text_frame.paragraphs:
              for run in paragraph.runs:
                  run.font.size = Pt(11)

  prs.save(pptpath) #this overwrites
  print('Presentation saved')

def ql_3band_batch(folder,outshp,outfolder):
  imglist = getfiles(folder,'.tif')
  for imgpath in imglist:
      name = Path(imgpath).stem + '.png'
      outpath = os.path.join(outfolder,name)
      ql_3band(outshp,imgpath,outpath)

def add_slides_batch(img_folder,pptpath):
  imglist = getfiles(img_folder,'.png')

  prs = Presentation(pptpath)
  title_only_slide_layout = prs.slide_layouts[5]

  for i in imglist:
      print(i)
      slide = prs.slides.add_slide(title_only_slide_layout)
      title = slide.shapes.title
      title.width = Cm(35)
      title.height = Cm(3.3)
      title.top = Cm(0)
      title.left = Cm(0)

      #Add title
      name = Path(i).stem
      title.text = name

      slide.shapes.add_picture(
          i, left=Cm(12.78), top=Cm(3.7), width=Cm(12.24), height=None
      )
  prs.save(pptpath)

def inset_map(bc_path,fire_perim,outpath):
  # Define the bounding box for British Columbia (lonmin, lonmax, latmin, latmax)
  bbox = [-139, -114.75, 47.5, 60]


  # Coordinates of cities/towns (latitude, longitude)
  cities = {
      'Vancouver': (49.2827, -123.1207),
      'Kamloops': (50.6761, -120.3408),
      'Prince George':(53.9170,-122.7494),
      'Fort St John': (56.2464,-120.8476),
      'Prince Rupert': (54.3125,-130.3054),
      'Williams Lake': (52.1284,-122.1302)

  }

  # Create a map
  plt.figure(figsize=(4,4))
  ax = plt.axes(projection=ccrs.AlbersEqualArea(central_longitude=-126, central_latitude=54))
  ax.set_extent(bbox, crs=ccrs.PlateCarree())

  # Add land and coastline features
  ax.add_feature(cfeature.LAND, edgecolor='black', facecolor='lightgrey',linewidth=0.1)
  ax.add_feature(cfeature.COASTLINE, linewidth=0.1)

  # Load the British Columbia boundary data using GeoPandas
  ## Thank you GeoBC! https://catalogue.data.gov.bc.ca/dataset/province-of-british-columbia-boundary-terrestrial
  bc_boundary = geopandas.read_file(bc_path)

  # Plot the British Columbia boundary using Cartopy's geopandas tools
  ax.add_geometries(bc_boundary['geometry'], crs=ccrs.PlateCarree(), edgecolor='black', facecolor='green',linewidth=0.1)

  # Add gridlines
  ax.gridlines(draw_labels=True, linestyle='--', color='grey')

  # Add fire boundary
  fire_boundary = geopandas.read_file(fire_perim)
  fire_boundary["centroid"] = fire_boundary["geometry"].centroid
  lat = fire_boundary["centroid"].y
  lon = fire_boundary["centroid"].x
  ax.plot(lon,lat, marker='*', color='purple', markersize=5, transform=ccrs.PlateCarree())
  ax.text(lon + 0.5, lat, 'Fire Location', color='purple', fontsize=8, transform=ccrs.PlateCarree())

  # Add cities/towns
  for city, (lat, lon) in cities.items():
      ax.plot(lon, lat, marker='o', color='black', markersize=3, transform=ccrs.PlateCarree())
      ax.text(lon + 0.5, lat, city, color='black', fontsize=7, transform=ccrs.PlateCarree())

  plt.savefig(outpath, bbox_inches='tight', dpi=300)
  plt.close()

# Calculates burn area by severity class
def zonal_barc(barcpath,firepath,outpath):
  def burnsev_name(x):
      if x == 0:
          return('Unknown')
      elif x == 1:
          return('Unburned')
      elif x == 2:
          return('Low')
      elif x == 3:
          return('Medium')
      elif x == 4:
          return('High')
      elif x == 5:
          return('Water')
      else:
          print("Wrong burn severity value!")


  #need to clip to fire perimeter a little tighter than what gee outputs
  basename = barcpath[:-4]
  barcpath_clip = basename + '_clip.tif'

  gdal.Warp(barcpath_clip,barcpath,cutlineDSName=firepath,cropToCutline=True)

  dat = gdal.Open(barcpath_clip)
  band = dat.GetRasterBand(1).ReadAsArray()

  x1,px,x2,x3,x4,x5 = dat.GetGeoTransform()

  nodatavalue = 9
  vals, counts = np.unique(band[band != nodatavalue], return_counts=True)

  df = pd.DataFrame(
      {'class':vals,
        'px_count':counts})


  a = df['px_count'].sum()
  df['perc' ] = (df['px_count'] / a)*100
  df['area_m2'] = df['px_count']*(px*px)
  df['area_ha'] = df['area_m2']*0.0001

  #round to 1 decimal place
  df = df.round(1)
  df['burn_sev'] = df['class'].apply(burnsev_name)

  #reorder columns
  df = df[['class','burn_sev','px_count','area_m2','area_ha','perc']]
  df_out = df[['class','burn_sev','area_ha','perc']]

  #print(df) #debug
  df_out.to_csv(outpath)
  return(df_out)

def aoionly(img):
  return(img.updateMask(poly_mask))



---



### **Part 2: Google Earth Engine authentication and initialization**

Authenticate and intialize GEE. After running the cell below, a sign-in window will pop-up. Please follow prompts to authenticate (sign-in, continue and continue).

In [5]:
#Authenticate gee
ee.Authenticate()

 Please note, the **Project ID** may be something other than the project name (ex. burn-severity-2024) and may contain additional numbers (ex. burn-severity-2024-456181). Make sure to copy the actual **Project ID** and enter it in the cell below.

In [6]:
# Initialize with a google cloud project
project = 'wlbr-2025'
ee.Initialize(project=project)



---



### **Part 3: Fire perimeter import and visualization**

The code below will load in a custom polygon shapefile or download the current fire perimeter dataset from BC Wildfire. If the file path in the fire_shp variable exists, then the code will default to that dataset.

In [10]:
# Define fire year (options:'current','2025')
fire_year = 'current'

# Get fire perimeter file (either user defined) or pull from BC Wildfire
# Open fires shapefile if exists
fires_shp = '/content/perims.shp'
if os.path.exists(fires_shp):
  print('Using user specified perimeter file')
else:
  if fire_year =='current':
    print('Downloading BC Wildfire current fire perimeter file')
    url = 'https://pub.data.gov.bc.ca/datasets/cdfc2d7b-c046-4bf0-90ac-4897232619e1/prot_current_fire_polys.zip'
    zipname = "prot_current_fire_poly.zip"

  elif fire_year == '2025':
    print('Downloading BC Wildfire historic fire perimeters file (2025)')
    url = 'https://coms.api.gov.bc.ca/api/v1/object/50017db4-fc62-4494-8fb1-e1a668726853'
    zipname = "prot_historic_fire_polys_faib.zip"

  response = requests.get(url)
  if response.status_code == 200:
      with open(zipname, 'wb') as file:
          file.write(response.content)
      print("File downloaded successfully")
  else:
      print(f"Failed to download file. Status code: {response.status_code}")

  outfolder = '/content/'+zipname.rsplit('.')[0]

  with zipfile.ZipFile(zipname, 'r') as zip_ref:
      zip_ref.extractall(outfolder)

  fires_shp = getfiles(outfolder,'.shp')[0]
  print('Fire perimeter file: ',fires_shp)

File downloaded successfully
Fire perimeter file:  /content/prot_current_fire_poly/prot_current_fire_polys.shp


In [11]:
import warnings
from google.colab import data_table
warnings.filterwarnings("ignore")
data_table.enable_dataframe_formatter()

# Visualize in table format
fires = geemap.shp_to_ee(fires_shp)
fires_df = geopandas.read_file(fires_shp)
fires_df_tbl = fires_df.drop(columns=['geometry'], axis=1, inplace=False)
#fires_df_tbl = fires_df_tbl[(fires_df_tbl['FIRE_STAT']=='Out') & (fires_df_tbl['FIRE_SZ_HA']>=100)] #uncomment for fires that are out and >= 100 ha
fires_df_tbl

,OBJECTID,FIRE_YEAR,FIRE_NUM,VERSN_NUM,FIRE_SZ_HA,SOURCE,TRACK_DATE,LOAD_DATE,FIRE_STAT,FIRE_LINK,FEATURE_CD
0,26803168,2026,G70322,2026051201,206.8,Non-corrected airborne GPS,2026-05-12,2026-05-13,Out,https://wildfiresituation.nrs.gov.bc.ca/incide...,JA70003000
1,26803169,2026,G70362,2026052101,159.6,Non-corrected airborne GPS,2026-05-21,2026-05-21,Out,https://wildfiresituation.nrs.gov.bc.ca/incide...,JA70003000
2,26803170,2026,G80206,2026050301,3.3,Non-corrected airborne GPS,2026-05-03,2026-05-14,Out,https://wildfiresituation.nrs.gov.bc.ca/incide...,JA70003000
3,26803171,2026,G80347,2026052801,13.8,Non-corrected ground GPS,2026-05-28,2026-05-29,Out,https://wildfiresituation.nrs.gov.bc.ca/incide...,JA70003000
4,26803172,2026,G80457,2026053101,1.2,Non-corrected ground GPS,2026-05-31,2026-06-03,Out,https://wildfiresituation.nrs.gov.bc.ca/incide...,JA70003000
...,...,...,...,...,...,...,...,...,...,...,...
90,26803205,2026,R50553,2026061501,30.2,Non-corrected ground GPS,2026-06-15,2026-06-16,Being Held,https://wildfiresituation.nrs.gov.bc.ca/incide...,JA70003000
91,26803206,2026,R90157,2026042901,8.2,Non-corrected ground GPS,2026-04-29,2026-04-30,Out,https://wildfiresituation.nrs.gov.bc.ca/incide...,JA70003000
92,26803207,2026,V10108,2026050401,61.6,Non-corrected airborne GPS,2026-05-04,2026-05-06,Under Control,https://wildfiresituation.nrs.gov.bc.ca/incide...,JA70003000
93,26803208,2026,V10212,2026050401,1.4,Unknown,2026-05-04,2026-06-17,Out,https://wildfiresituation.nrs.gov.bc.ca/incide...,JA70003000




---



### **Part 4: Individual fire perimeter selection (1 fire perimeter)**

Select one fire perimeter by fire number. Please make sure that the **fieldname** variable matches your dataset.

In [12]:
# Now select one fire (in the test data, there's only one fire perimeter)
firenumber = 'C10467' #change fire name
fieldname = 'FIRE_NUM' #unique firenumber field, change if needed

# First check if the firenumber exists in the shapefile provided
firelist = fires_df[fieldname].tolist()

if firenumber not in firelist:
  print('Selected fire number:',firenumber)
  print('Available fire numbers: ',firelist)
  raise ValueError('Fire number not in fire list. Typo?')

# Create output folder
if not os.path.exists(firenumber):
  os.mkdir(firenumber)

# Save a copy of the fire perimeter
vector_folder = os.path.join(firenumber,'vectors')
if not os.path.exists(vector_folder):
  os.mkdir(vector_folder)

outshp = os.path.join(vector_folder,firenumber+'.shp')
fires_df_sub = fires_df[fires_df[fieldname]==firenumber]
fires_df_sub.to_file(outshp,driver='ESRI Shapefile')

# Load in the single perimeter
poly = geemap.shp_to_ee(outshp)

# Create raster mask to reduce extent of image collections
# Function aoionly in functions
poly_buf = poly.geometry().buffer(500).bounds()
poly_mask = ee.Image.constant(1).clip(poly_buf).selfMask()

Now visualize spatially.

In [13]:
Map = geemap.Map()
Map.addLayer(poly,{},'Fire Perimeter')
Map.centerObject(poly,zoom=12)
Map

Map(center=[53.34178194140193, -124.19130913303287], controls=(WidgetControl(options=['position', 'transparent…



---



### **Part 5: Sensor selection**

 The sensor options are as follows: Sentinel-2a/b MSI (S2), Landsat-8 OLI (L8) or Landsat-9 OLI-2 (L9). The data from the above sensors are the surface reflectance / bottom of atmosphere processing level. Pre- and post-fire imagery wil be selected from the same sensor.

In [30]:
dattype = 'S2' #change sensor here

if dattype == 'S2':
    col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').map(aoionly)
    cld_field =  'CLOUDY_PIXEL_PERCENTAGE'
    print('Selected S2 SR')
elif dattype == 'L9':
    col = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').map(aoionly)
    cld_field = 'CLOUD_COVER'
    print('Selected L9 SR')
elif dattype == 'L8':
    col = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(aoionly)
    cld_field = 'CLOUD_COVER'
    print('Selected L8 SR')
elif dattype == 'L8_TOA':
    col = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA').map(aoionly)
    cld_field = 'CLOUD_COVER'
    print('Selected L8 TOA')
elif dattype == 'L9_TOA':
    col = ee.ImageCollection('LANDSAT/LC09/C02/T1_TOA').map(aoionly)
    cld_field = 'CLOUD_COVER'
    print('Selected L9 TOA')
else:
    raise ValueError('Wrong data type selected. Choose S2, L9 or L8.')


Selected S2 SR


**Visualization parameters**: choose the type of visualization you would like to use. Default is false-colour infrared ('nir') but true-color ('tc') and shortwave-infrared ('swir') are also available.

In [31]:
# Define visualization parameters
vis_type = 'tc'

# Dictionary with band combinations
vis_dict = {'L8':{'nir':['SR_B5', 'SR_B4', 'SR_B3'],
                  'tc':['SR_B4','SR_B3','SR_B2'],
                  'swir':['SR_B7','SR_B6','SR_B4']},
            'L9':{'nir':['SR_B5', 'SR_B4', 'SR_B3'],
                  'tc':['SR_B4','SR_B3','SR_B2'],
                  'swir':['SR_B7','SR_B6','SR_B4']},
            'L8_TOA':{'nir':['B5', 'B4', 'B3'],
                  'tc':['B4','B3','B2'],
                  'swir':['B7','B6','B4']},
            'L9_TOA':{'nir':['B5', 'B4', 'B3'],
                  'tc':['B4','B3','B2'],
                  'swir':['B7','B6','B4']},
            'S2':{'nir':['B8', 'B4', 'B3'],
                  'tc':['B4','B3','B2'],
                  'swir':['B12','B8','B4']}}

bands = vis_dict[dattype][vis_type] #or provide a list of 3 bands

vis = {
  'min': 0,
  'max': 0.3,
  'gamma':1.5,
  'bands': bands,
};



---



### **Part 6: Pre-fire image search, visualization and selection**

Select the time-interval for the pre-fire image search (startdate_pre and enddate_pre). The imagery should be selected either from year of the fire or the year before. The optimal window for image selection is July 1st to September 30. If good quality imagery isn't available, then search previous year's imagery. To avoid phenology differences, it is best to select pre- and post-fire imagery that is as seasonally consistent as possible. For example both and pre- and post-fire imagery acquired in early August.

The script below will search the archive for images acquired during the defined time interval (not inclusive of the enddate), overlapping with the fire perimeter, and below the defined cloud cover threshold. The resulting images are then mosaicked by image acquisition day (along track). If you are searching a wide range time range, it is best to reduce the cloud threshold (cld_thr_pre) to 40 or less.


In [38]:
# Find pre-fire imagery
startdate_pre = '2026-06-01'
enddate_pre = '2026-06-05'
cld_thr_pre = 20 #this is the cloud cover threshold from the scene metadata

before = col.filterDate(startdate_pre,enddate_pre).filterBounds(poly).filter(ee.Filter.lt(cld_field,cld_thr_pre))

before_list = before.toList(before.size().getInfo())
pre_mosaic_col = runDateMosaic(before_list)

if (dattype == 'L5')|(dattype == 'L7')|(dattype == 'L8')|(dattype == 'L9'):
  pre_mosaic_col = pre_mosaic_col.map(apply_scale_factors_ls)
elif (dattype == 'S2'):
  pre_mosaic_col = pre_mosaic_col.map(apply_scale_factors_s2)
else:
  pass

print('Found',pre_mosaic_col.size().getInfo(),'dates')

Found 1 dates


**Pre-fire image map display:** display the imagery found.

In [40]:
# Add pre-fire images to pre-fire map
before_map = geemap.Map()

before_size = pre_mosaic_col.size().getInfo()
before_mosaic_list = pre_mosaic_col.toList(before_size)
before_size

for i in range(0,before_size):
  b = before_mosaic_list.get(i)
  date = b.getInfo()['properties']['system:index']
  before_map.addLayer(ee.Image(b), vis, date)
  print(date)

# Add fire polygon
style = {'color': 'white', 'width': 2, 'lineType': 'solid', 'fillColor': '00000000'}
before_map.addLayer(poly.style(**style),{},'poly')
before_map.centerObject(poly,zoom=12)
before_map

2026-06-01


Map(center=[53.34178194140193, -124.19130913303287], controls=(WidgetControl(options=['position', 'transparent…

In the top right corner of the map, select the option to list all the layers and toggle on and off the dates to inspect the imagery. Find a high quality image, free of cloud, smoke, as well as snow and take note of the date(yyyy-mm-dd). You can copy and paste the date from the output of the cell above. If there's no clear imagery available for the date you range you defined you can try a different date range or go back and select a different sensor (dattype).



---



### **Part 7: Post-fire image search, visualization and selection**

Now select the time-interval for the post-fire image search (startdate_post and enddate_post). As with the pre-fire imagery, the optimal window for image selection is July 1st to September 30. However, that will not be possible, especially for the fires later in the season. If possible, try to select imagery from before October 15th.

In [34]:
# Find post-fire images
startdate_post = '2026-06-01'
enddate_post = '2026-06-18' #not inclusive of the end date
cld_thr_post = 100

after = col.filterDate(startdate_post,enddate_post).filterBounds(poly).filter(ee.Filter.lt(cld_field,cld_thr_post))

after_list = after.toList(after.size().getInfo())
post_mosaic_col = runDateMosaic(after_list)

if (dattype == 'L5')|(dattype == 'L7')|(dattype == 'L8')|(dattype == 'L9'):
  post_mosaic_col = post_mosaic_col.map(apply_scale_factors_ls)
elif (dattype == 'S2'):
  post_mosaic_col = post_mosaic_col.map(apply_scale_factors_s2)
else:
  pass

print('Found',post_mosaic_col.size().getInfo(),'scenes')

Found 8 scenes


**Post-fire image map display**: display the post-fire imagery in a new map.

In [35]:
# Add post-fire images to post-fire map
after_map = geemap.Map()

post_size = post_mosaic_col.size().getInfo()
post_mosaic_list = post_mosaic_col.toList(post_size)

for i in range(0,post_size):
  b = post_mosaic_list.get(i)
  date = b.getInfo()['properties']['system:index']
  after_map.addLayer(ee.Image(b), vis, date)
  print(date)

# Add fire polygon
style = {'color': 'white', 'width': 2, 'lineType': 'solid', 'fillColor': '00000000'}
after_map.addLayer(poly.style(**style),{},'poly')
after_map.centerObject(poly,zoom=12)
after_map


2026-06-01
2026-06-03
2026-06-04
2026-06-06
2026-06-09
2026-06-11
2026-06-14
2026-06-16


Map(center=[53.34178194140193, -124.19130913303287], controls=(WidgetControl(options=['position', 'transparent…

Also take note of the post-fire image selected. If good quality post-fire imagery is not available consider changing the date range or the sensor.



---



### **Part 8: BARC mapping**

The BARC burn severity raster raster will be created below. Change the dates for the pre_mosaic_date and post_mosaic_date variables. The BARC raster is generated by: (1) calculating pre- and post-NBR, (2) calculating dNBR by subtracting postNBR from preNBR, (3) Applying a scaling equation to dNBR and (4) Applying thresholds to create a 4-class burn severity raster (1-unburned, 2-low, 3-medium, 4-high).

In [41]:
# Calculate NBR, dNBR and generate BARC map.
pre_mosaic_date = '2026-06-01' #enter pre-fire image date here
post_mosaic_date = '2026-06-16' #enter post_fire image date here

# Select pre-image and post-image
pre_col = pre_mosaic_col.filter(ee.Filter.inList("system:index",ee.List([pre_mosaic_date])))
pre_img = ee.Image(pre_col.toList(1).get(0))

if pre_col.size().getInfo() != 1:
  raise ValueError("Didn't select 1 pre-fire image date. Check pre_mosaic_date!")

post_col = post_mosaic_col.filter(ee.Filter.inList("system:index", ee.List([post_mosaic_date])))
post_img = ee.Image(post_col.toList(1).get(0))

if post_col.size().getInfo() != 1:
  raise ValueError("Didn't select 1 post-fire image date. Check post_mosaic_date!")

# Calculate NBR
if dattype.startswith('S2'):
    pre_nbr = NBR_S2(pre_img)
    post_nbr = NBR_S2(post_img)
elif (dattype=='L8')|(dattype=='L9'):
    pre_nbr = NBR_Landsat(pre_img,dattype)
    post_nbr = NBR_Landsat(post_img,dattype)
else:
    pre_nbr = NBR_Landsat_TOA(pre_img,dattype)
    post_nbr = NBR_Landsat_TOA(post_img,dattype)

# Calculate dNBR
# dNBR theoretical range of values is -2 to 2, may have some outliers
dNBR = pre_nbr.subtract(post_nbr).rename('dNBR')
minMax = dNBR.reduceRegion(
      reducer=ee.Reducer.minMax(),
      geometry=poly,
      scale=20,
      maxPixels=1e13)
print(minMax.getInfo())

#dNBRx1000 and integer
dNBR_1000 = dNBR.expression('dNBR * 1000',{'dNBR': dNBR.select('dNBR')}).rename('dNBR_export')
dNBR_1000_thr = dNBR_1000.expression("(dNBR_export < -2000) ? -2000 "
                                ": (dNBR_export >= 2000) ? 2000 "
                                ": dNBR_export",
                                 {'dNBR_export': dNBR_1000.select('dNBR_export')})

dNBR_export = dNBR_1000_thr.round().toInt16()
minMax = dNBR_export.reduceRegion(
      reducer=ee.Reducer.minMax(),
      geometry=poly,
      scale=20,
      maxPixels=1e13)
print(minMax.getInfo())

# Scale dNBR to integer
dNBR_scaled = dNBR.expression('(dNBR * 1000 + 275)/5',{'dNBR': dNBR.select('dNBR')}).rename('dNBR_scaled')

# Create BARC4 layer
classes = dNBR_scaled.expression("(dNBR_scaled >= 187) ? 4 "
                                ": (dNBR_scaled >= 110) ? 3 "
                                ": (dNBR_scaled >= 76) ? 2 "
                                ": 1",{'dNBR_scaled': dNBR_scaled.select('dNBR_scaled')})

# Clip to fire perimeter
classes_clipped_nomsk = classes.clip(poly)

#Create BARC256 layer
dNBR_scaled_round = dNBR_scaled.round().rename('dNBR_scaled_round')

minMax = dNBR_scaled_round.reduceRegion(
      reducer=ee.Reducer.minMax(),
      geometry=poly,
      scale=20,
      maxPixels=1e13)
print(minMax.getInfo())

# Set values
dNBR_scaled_round_set = dNBR_scaled_round.expression("(dNBR_scaled_round <= 1) ? 1 "
                                ": (dNBR_scaled_round >= 254) ? 254 "
                                ": dNBR_scaled_round",
                                 {'dNBR_scaled_round': dNBR_scaled_round.select('dNBR_scaled_round')}).rename('dNBR_scaled_round_set')

# BARC256 range of values should be 1 to 254
# (0 withheld for no data, 255 for cloud masking)
BARC256 = dNBR_scaled_round_set.toUint8().clip(poly).rename('BARC256')
minMax = BARC256.reduceRegion(
      reducer=ee.Reducer.minMax(),
      geometry=poly,
      scale=20,
      maxPixels=1e13)
print(minMax.getInfo())

# Get scene ids to save
d = ee.Date(pre_mosaic_date)
pre_filt = before.filterDate(d,d.advance(1,'day'))
pre_scenes_ids = pre_filt.aggregate_array('system:index').getInfo()
print('Pre-fire scene IDs:',pre_scenes_ids)

d1 = ee.Date(post_mosaic_date)
post_filt = after.filterDate(d1,d1.advance(1,'day'))
post_scenes_ids = post_filt.aggregate_array('system:index').getInfo()
print('Post-fire scene IDs:',post_scenes_ids)
print("BARC Processing Complete")

{'dNBR_max': 1.2773043921845941, 'dNBR_min': -0.3927071185135701}
{'dNBR_export_max': 1277, 'dNBR_export_min': -393}
{'dNBR_scaled_round_max': 310, 'dNBR_scaled_round_min': -24}
{'BARC256_max': 254, 'BARC256_min': 1}
Pre-fire scene IDs: ['20260601T191911_20260601T192612_T10UDE']
Post-fire scene IDs: ['20260616T191909_20260616T192847_T10UDE', '20260616T193841_20260616T193835_T10UDE']
BARC Processing Complete




---



Mask out water in the BARC layer and set it to 1 (unburned).

In [42]:
# Read in ESA 10m landcover (https://gee-community-catalog.org/projects/esa_iq/)
waterMask=True
if waterMask:
  print('Masking water')
  esri_lc = ee.ImageCollection('ESA/WorldCover/v200').filterBounds(poly).map(aoionly)
  esri_lc_last = ee.Image(esri_lc.sort('system:time_start',False).first())
  if dattype.startswith('S2'):
    px = 20
  else:
    px = 30

  water = esri_lc_last.eq(80).Not().reproject(crs=classes_clipped_nomsk.projection(),scale=px) #reproject to barc
  classes_clipped = classes_clipped_nomsk.mask(water).unmask(5).clip(poly) #setting masked water to 5
else:
  print('Not masking water')
  classes_clipped = classes_clipped_nomsk

Masking water




---



### **Part 9: Final visualization for quality control**

This script is the final visualization of pre- and post-fire imagery as well as the BARC map. To assess the quality, toggle the layers and zoom into different regions to make sure that the pre- and post-fire imagery is free of clouds, smoke, haze or active fire.

In [43]:
#Visualize pre- and post-fire imagery with BARC
barc_map = geemap.Map()
barc_map.addLayer(ee.Image(pre_img), vis, pre_mosaic_date)
barc_map.addLayer(ee.Image(post_img),vis, post_mosaic_date)

palette = ['000000','grey', 'yellow', 'orange','red','blue']
keys = ['Unknown','Unburned','Low','Medium','High','Water']
colors = [(0, 0, 0),(128, 128, 128),(255, 255, 0),(255, 165, 0), (255, 0, 0),(0, 0, 255)]

barc_vis = {'min':0,'max':5,'palette':palette}
barc_map.addLayer(ee.Image(classes_clipped),barc_vis,'barc')
barc_map.add_legend(keys=keys,colors=colors,position='bottomright')

# barc256_vis = {'min':1,'max':254}
# barc_map.addLayer(ee.Image(BARC256),barc256_vis,'barc256')

# water_vis = {'min':0,'max':1}
# barc_map.addLayer(water,water_vis,'water')

style = {'color': 'white', 'width': 2, 'lineType': 'solid', 'fillColor': '00000000'}
barc_map.addLayer(poly.style(**style),{},'poly')

barc_map.centerObject(poly,zoom=12)
barc_map

Map(center=[53.341781941401045, -124.19130913303384], controls=(WidgetControl(options=['position', 'transparen…

### **Part 10: Image download**

This script will download a BARC raster clipped to the extent of the fire perimeter polygon, as well as pre- and post-fire imagery (true color and swir RGB, 8-bit). The search criteria and scene ids will also be exported as json and text files. The root folder will be the fire number.

In [44]:
# Export
crs_out = 'EPSG:3005'
outfolder = firenumber #root folder
pre_date = pre_mosaic_date.replace('-','')
post_date = post_mosaic_date.replace('-','')

folders = [
    'barc',
    'barc256',
    'dNBR',
    'pre_truecolor_8bit',
    'post_truecolor_8bit',
    'pre_swir_8bit',
    'post_swir_8bit']

filenames = [
    f'BARC_{firenumber}_{pre_date}_{post_date}_{dattype}',
    f'BARC256_{firenumber}_{pre_date}_{post_date}_{dattype}',
    f'dNBR_{firenumber}_{pre_date}_{post_date}_{dattype}',
    f'{dattype}_{pre_date}_truecolor_pre_8bit',
    f'{dattype}_{post_date}_truecolor_post_8bit',
    f'{dattype}_{pre_date}_swir_pre_8bit',
    f'{dattype}_{post_date}_swir_post_8bit']

for folder_name in folders:
    folder_path = os.path.join(outfolder, folder_name)
    if os.path.exists(folder_path):
        print(f"{folder_path} exists. Deleting.")
        shutil.rmtree(folder_path)
    os.makedirs(folder_path)

# Define spatial resolution
if dattype.startswith('S2'):
  px = 20
else:
  px = 30

## Define tiling rules
poly_area = round(poly.geometry().area(1).divide(10000).getInfo(),1)
print(poly_area)
print('Fire Area:',poly_area,'hectares')

if poly_area < 10000:
    n = 2
elif poly_area >= 10000 and poly_area < 100000:
    n = 3
elif poly_area >= 100000 and poly_area < 400000:
    n = 4
elif poly_area >= 400000 and poly_area < 1000000:
    n = 5
else:
    n = 6

print('Number of tiles: ' + str(n*n))

#export pre and post rgbs, tile to avoid pixel limit issues.
footprint = poly.geometry().bounds().getInfo()
grids = grid_footprint(footprint,n,n)

for i in range(0,len(grids)):
  roi = grids[i]
  ## Export BARC
  name = filenames[0] +'_' + str(i) + '.tif'
  barc_filename = os.path.join(outfolder,'barc',name)
  geemap.ee_export_image(classes_clipped.unmask(9).clip(roi), filename=barc_filename, scale=px, file_per_band=False,crs=crs_out)
  ras = gdal.Open(barc_filename,gdal.GA_Update)
  dat = ras.GetRasterBand(1)
  dat.SetNoDataValue(9)
  ras = None
  dat = None

  ## Export BARC256
  name = filenames[1] +'_' + str(i) + '.tif'
  barc256_filename = os.path.join(outfolder,'barc256',name)
  geemap.ee_export_image(BARC256.unmask(0).clip(roi), filename=barc256_filename, scale=px, file_per_band=False,crs=crs_out)
  ras = gdal.Open(barc256_filename,gdal.GA_Update)
  dat = ras.GetRasterBand(1)
  dat.SetNoDataValue(0)
  ras = None
  dat = None

  ## Export dNBR
  name = filenames[2] +'_' + str(i) + '.tif'
  dNBR_filename = os.path.join(outfolder,'dNBR',name)
  geemap.ee_export_image(dNBR_export.clip(roi), filename=dNBR_filename, scale=px, file_per_band=False,crs=crs_out)

  ## Export 8-bit truecolor images
  #pre, truecolor
  name = filenames[3] +'_' + str(i) + '.tif'
  pre_tc_8bit_path = os.path.join(outfolder,'pre_truecolor_8bit',name)
  if dattype.startswith('S2'):
      viz = {'bands': ['B4', 'B3', 'B2'], 'min': -0.01, 'max':0.3,'gamma':1.5}
      geemap.ee_export_image(pre_img.clip(roi).visualize(**viz), filename=pre_tc_8bit_path, scale=10, file_per_band=False,crs=crs_out)
  elif (dattype.startswith('L8')) | (dattype.startswith('L9')):
      viz = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': -0.01, 'max':0.3,'gamma':1.5}
      geemap.ee_export_image(pre_img.clip(roi).visualize(**viz), filename=pre_tc_8bit_path, scale=30, file_per_band=False,crs=crs_out)
  else:
      pass

  #post, truecolor
  name = filenames[4] +'_' + str(i) + '.tif'
  post_tc_8bit_path = os.path.join(outfolder, 'post_truecolor_8bit', name)
  if dattype.startswith('S2'):
      viz = {'bands': ['B4', 'B3', 'B2'], 'min': -0.01, 'max':0.3,'gamma':1.5}
      geemap.ee_export_image(post_img.clip(roi).visualize(**viz), filename=post_tc_8bit_path, scale=10, file_per_band=False,crs=crs_out)
  elif (dattype.startswith('L8')) | (dattype.startswith('L9')):
      viz = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': -0.01, 'max':0.3,'gamma':1.5}
      geemap.ee_export_image(post_img.clip(roi).visualize(**viz), filename=post_tc_8bit_path, scale=30, file_per_band=False,crs=crs_out)
  else:
      pass

  ## Export swir too
  # pre, swir
  name = filenames[5] +'_' + str(i) + '.tif'
  pre_sw_8bit_path = os.path.join(outfolder,'pre_swir_8bit',name)
  if dattype.startswith('S2'):
      viz = {'bands': ['B12', 'B8', 'B4'], 'min': -0.01, 'max':0.3,'gamma':1.5}
      geemap.ee_export_image(pre_img.clip(roi).visualize(**viz), filename=pre_sw_8bit_path, scale=10, file_per_band=False,crs=crs_out)
  elif (dattype.startswith('L8')) | (dattype.startswith('L9')):
      viz = {'bands': ['SR_B6', 'SR_B5', 'SR_B4'], 'min': -0.01, 'max':0.3,'gamma':1.5}
      geemap.ee_export_image(pre_img.clip(roi).visualize(**viz), filename=pre_sw_8bit_path, scale=30, file_per_band=False,crs=crs_out)
  else:
      pass

  #post, swir
  name = filenames[6] +'_' + str(i) + '.tif'
  post_sw_8bit_path = os.path.join(outfolder,'post_swir_8bit',name)
  if dattype.startswith('S2'):
      viz = {'bands': ['B12', 'B8', 'B4'], 'min': -0.01, 'max':0.3,'gamma':1.5}
      geemap.ee_export_image(post_img.clip(roi).visualize(**viz), filename=post_sw_8bit_path, scale=10, file_per_band=False,crs=crs_out)
  elif (dattype.startswith('L8')) | (dattype.startswith('L9')):
      viz = {'bands': ['SR_B6', 'SR_B5', 'SR_B4'], 'min': -0.01, 'max':0.3,'gamma':1.5}
      geemap.ee_export_image(post_img.clip(roi).visualize(**viz), filename=post_sw_8bit_path, scale=30, file_per_band=False,crs=crs_out)
  else:
      pass

#mosaic all
def mosaic_all(subfolder,outfilename):
  filelist = getfiles(os.path.join(outfolder,subfolder),'.tif')
  out = os.path.join(outfolder,subfolder,outfilename)
  gdal.Warp(out,filelist)
  for file in filelist: os.remove(file) #delete tiles
  print(subfolder,'complete')

for folder_name, file_name in zip(folders,filenames):
  file_name = file_name + '.tif'
  mosaic_all(folder_name,file_name)

#Export a dictionary with metadata info
searchd = {'Id':firenumber,'sensor':dattype,
               'cld_pre':cld_thr_pre,'pre_T1':startdate_pre,'pre_T2':enddate_pre,
               'cld_post':cld_thr_post,'post_T1':startdate_post,'post_T2':startdate_post,
               'pre_mosaic_date':pre_mosaic_date,'pre_scenes':pre_scenes_ids,
               'post_mosaic_date':post_mosaic_date,'post_scenes':post_scenes_ids}

params = os.path.join(outfolder,'search_params.txt')
with open(params, 'w') as f:
    for key, value in searchd.items():
        f.write('%s:%s\n' % (key, value))

#write search parameters to the folder as json for use later
output_json = os.path.join(outfolder,'search_params.json')
with open(output_json, 'w') as json_file:
    json.dump(searchd, json_file, indent=4)


1767.6
Fire Area: 1767.6 hectares
Number of tiles: 4
Generating URL ...
Please wait ...
Data downloaded to /content/C10467/barc/BARC_C10467_20260601_20260616_S2_0.tif
Generating URL ...
Please wait ...
Data downloaded to /content/C10467/barc256/BARC256_C10467_20260601_20260616_S2_0.tif
Generating URL ...
Please wait ...
Data downloaded to /content/C10467/dNBR/dNBR_C10467_20260601_20260616_S2_0.tif
Generating URL ...
Please wait ...
Data downloaded to /content/C10467/pre_truecolor_8bit/S2_20260601_truecolor_pre_8bit_0.tif
Generating URL ...
Please wait ...
Data downloaded to /content/C10467/post_truecolor_8bit/S2_20260616_truecolor_post_8bit_0.tif
Generating URL ...
Please wait ...
Data downloaded to /content/C10467/pre_swir_8bit/S2_20260601_swir_pre_8bit_0.tif
Generating URL ...
Please wait ...
Data downloaded to /content/C10467/post_swir_8bit/S2_20260616_swir_post_8bit_0.tif
Generating URL ...
Please wait ...
Data downloaded to /content/C10467/barc/BARC_C10467_20260601_20260616_S2_1.t

### **Part 11: Quicklooks and area burned by severity class**

The script below creates and exports quicklook images, burn severity classes summaries and maps. You should be able to run as is.

In [45]:
# Summary Section
qcdir = os.path.join(firenumber,'QC')
if os.path.exists(qcdir):
  print(qcdir,'exists. Deleting.')
  shutil.rmtree(qcdir)
os.mkdir(qcdir)

# Create powerpoint presentation
pptpath = os.path.join(qcdir,firenumber+'.pptx')
create_ppt(pptpath)

bc_boundary ='burnSeverity/bc/BC_Boundary_Terrestrial_gcs_simplify.shp'

# Pre- and post-fire quicklooks (truecolor)
pre_tc_8bit_path = os.path.join(outfolder,folders[3],filenames[3]+'.tif')
pre_tc_ql = generate_ql(outshp,pre_tc_8bit_path,qcdir)

post_tc_8bit_path = os.path.join(outfolder,folders[4],filenames[4]+'.tif')
post_tc_ql = generate_ql(outshp,post_tc_8bit_path,qcdir)

# Pre- and post-fire quicklooks (swir)
pre_sw_8bit_path = os.path.join(outfolder,folders[5],filenames[5]+'.tif')
pre_sw_ql = generate_ql(outshp,pre_sw_8bit_path,qcdir)
post_sw_8bit_path = os.path.join(outfolder,folders[6],filenames[6]+'.tif')
post_sw_ql = generate_ql(outshp,post_sw_8bit_path,qcdir)

# BARC
name = Path(filenames[0]).stem + '.png'
barc_ql = os.path.join(qcdir,name)
barc_filename = os.path.join(outfolder,folders[0],filenames[0]+'.tif')
ql_barc(outshp,barc_filename,post_tc_8bit_path,barc_ql)

# Location map
name = firenumber + '_locmap.png'
map_ql = os.path.join(qcdir,name)

fire_perim = os.path.join(firenumber,'vectors',firenumber+'_gcs.shp')
if os.path.isfile(fire_perim):
    pass
else:
    fire_perim = os.path.join(firenumber,'vectors',firenumber+'.shp')

inset_map(bc_boundary,fire_perim,map_ql)

#add stats table
outpath = os.path.join(firenumber,'barc_stats.csv')
df = zonal_barc(barc_filename,outshp,outpath)

#Add slide to powerpoint
add_slide(pptpath,pre_tc_ql,post_tc_ql,barc_ql,map_ql,df)
add_slide(pptpath,pre_sw_ql,post_sw_ql,barc_ql,map_ql,df)



Presentation saved
Presentation saved




---



### **Part 12: Final export**

The folder will be zipped and saved under the Files tab. Right click and download the folder to your machine.

In [46]:
#Zip to download
zipped = firenumber + '.zip'
indata = firenumber

!zip -r {zipped} {indata}
files.download(zipped)

  adding: C10467/ (stored 0%)
  adding: C10467/QC/ (stored 0%)
  adding: C10467/QC/S2_20260601_truecolor_pre_8bit.png (deflated 0%)
  adding: C10467/QC/S2_20260616_truecolor_post_8bit.png (deflated 0%)
  adding: C10467/QC/BARC_C10467_20260601_20260616_S2.png (deflated 2%)
  adding: C10467/QC/C10467_locmap.png (deflated 6%)
  adding: C10467/QC/S2_20260616_swir_post_8bit.png (deflated 0%)
  adding: C10467/QC/C10467.pptx (deflated 0%)
  adding: C10467/QC/S2_20260601_swir_pre_8bit.png (deflated 0%)
  adding: C10467/barc/ (stored 0%)
  adding: C10467/barc/BARC_C10467_20260601_20260616_S2.tif (deflated 91%)
  adding: C10467/barc/BARC_C10467_20260601_20260616_S2_clip.tif (deflated 91%)
  adding: C10467/barc256/ (stored 0%)
  adding: C10467/barc256/BARC256_C10467_20260601_20260616_S2.tif (deflated 57%)
  adding: C10467/search_params.json (deflated 62%)
  adding: C10467/dNBR/ (stored 0%)
  adding: C10467/dNBR/dNBR_C10467_20260601_20260616_S2.tif (deflated 34%)
  adding: C10467/vectors/ (stored 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>